# Nominally significant univariate results

Read all univariate Cox results used for the reported
**With other primaries** (`with_other_primaries`) cohort and retain rows with nominal
significance (`p_value < 0.05`). This is an exploratory filter; `q_value`
remains in the table so multiplicity-adjusted significance can be assessed
separately.

The shared-canonical result file is preferred because it contains every
reported landmark in one table and is the source used by Figure 3. Legacy
per-landmark files are loaded only when the shared file is unavailable.

In [ ]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

COHORT = "with_other_primaries"
COHORT_LABEL = "With other primaries"
NOMINAL_ALPHA = 0.05
NEPC_PROJ_PATH = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS")
RUN_DIR = NEPC_PROJ_PATH / "survival_analysis" / f"local_runs_{COHORT}"

RESULT_FILENAME = "cox_agg_univariate_nobs_adjusted.csv"
SHARED_RESULT = RUN_DIR / "cox" / "landmark_shared" / RESULT_FILENAME
EXPORT_PATH = RUN_DIR / "cox" / "nominally_significant_univariate_results.csv"

print(f"Cohort: {COHORT_LABEL} ({COHORT})")
print(f"Run directory: {RUN_DIR}")

## Load every landmark

Using the shared file when present prevents double-counting the same models
from both shared and legacy per-landmark output directories.

In [ ]:
def load_univariate_results(run_dir: Path) -> tuple[pd.DataFrame, list[Path]]:
    shared_path = run_dir / "cox" / "landmark_shared" / RESULT_FILENAME
    if shared_path.exists():
        paths = [shared_path]
    else:
        paths = sorted((run_dir / "cox").glob(
            f"landmark_*/both/{RESULT_FILENAME}"
        ))

    if not paths:
        raise FileNotFoundError(
            f"No univariate result files found under {run_dir / 'cox'}. "
            "Run the univariate models first."
        )

    frames = []
    for path in paths:
        frame = pd.read_csv(path)
        if "landmark_days" not in frame.columns:
            match = re.search(r"landmark_(-?\\d+)", str(path.parent.parent))
            if match is None:
                raise ValueError(f"Could not infer landmark from {path}")
            frame.insert(0, "landmark_days", int(match.group(1)))
        frame["source_path"] = str(path)
        frames.append(frame)

    results = pd.concat(frames, ignore_index=True)
    required = {"landmark_days", "endpoint", "feature", "p_value"}
    missing = required - set(results.columns)
    if missing:
        raise ValueError(f"Univariate results are missing columns: {sorted(missing)}")

    results["p_value"] = pd.to_numeric(results["p_value"], errors="coerce")
    results.insert(0, "cohort", COHORT)
    return results, paths


univariate_results, source_paths = load_univariate_results(RUN_DIR)
print("Loaded:")
for path in source_paths:
    print(f"  {path}")
print(
    f"{len(univariate_results):,} rows across landmarks "
    f"{sorted(univariate_results['landmark_days'].dropna().unique().tolist())}"
)

## Filter at nominal significance

No stability or false-discovery-rate filter is applied here: every finite
`p_value < 0.05` row is retained.

In [ ]:
nominally_significant = (
    univariate_results.loc[
        univariate_results["p_value"].notna()
        & univariate_results["p_value"].lt(NOMINAL_ALPHA)
    ]
    .sort_values(["endpoint", "landmark_days", "p_value", "feature"])
    .reset_index(drop=True)
)

preferred_columns = [
    "cohort", "landmark_days", "endpoint", "feature", "lab_name",
    "feature_stat", "coverage", "n_patients_used", "n_events_used",
    "coef_feature", "hazard_ratio_per_sd", "ci_lower", "ci_upper",
    "p_value", "q_value", "note", "model_type", "source_path",
]
ordered_columns = [c for c in preferred_columns if c in nominally_significant.columns]
ordered_columns += [c for c in nominally_significant.columns if c not in ordered_columns]
nominally_significant = nominally_significant[ordered_columns]

summary = (
    nominally_significant
    .groupby(["endpoint", "landmark_days"], dropna=False)
    .size()
    .rename("n_nominally_significant")
    .reset_index()
)

print(
    f"Retained {len(nominally_significant):,} / {len(univariate_results):,} "
    f"rows with p < {NOMINAL_ALPHA}."
)
display(summary)

## Review all nominal hits

In [ ]:
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", 180,
):
    display(nominally_significant)

## Export the filtered table

In [ ]:
EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
nominally_significant.to_csv(EXPORT_PATH, index=False)
print(f"Wrote {len(nominally_significant):,} rows to {EXPORT_PATH}")